# Copernicus ERA5 Reanalysis via CDS

**Last updated:** 2026-05-09

---

## Learning objectives

1. Set up `cdsapi` credentials and access the Copernicus Climate Data Store (CDS)
2. Download ERA5 surface and pressure-level data for a custom region and time period
3. Open GRIB data with `earthkit.data`: same workflow as ECMWF Open Data
4. Compute a simple climatological baseline for Africa (2m temperature)
5. Understand when to use ERA5 vs real-time Open Data

---

## Introduction

**ERA5** is ECMWF's fifth-generation atmospheric reanalysis, covering **1940 to present** at 0.25°.  
It is available free of charge via the **Copernicus Climate Data Store (CDS)**.

ERA5 is the foundation for:
- Impact model training and validation (IbF, flood, drought)
- Climate baselines and anomaly detection
- Initialisation of regional/climate models

> **Credentials:** A free CDS account is required.  
> Register at [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu), then set your personal token in `~/.cdsapirc`:  
> ```
> url: https://cds.climate.copernicus.eu/api
> key: <YOUR-PERSONAL-ACCESS-TOKEN>
> ```

**Prerequisites:** `cdsapi`, `earthkit-data[cds]`, `earthkit-plots`, `xarray`  
**Run time:** ~10 minutes (downloads ~20 MB)  
**Data:** ERA5: 1 month, Africa bounding box


## 1) Setup & credentials

We use **earthkit.data** as the primary interface — the same API as for ECMWF Open Data (A01/A02).
Earthkit wraps `cdsapi` internally, so you only need one import pattern across all notebooks.

**Prerequisites:**
- CDS account (free): [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu)
- API key saved in `~/.cdsapirc` (shown at the bottom of your CDS profile page)

> **Native API:** `cdsapi` is the original CDS Python client and works directly without earthkit.
> We show it as an alternative in the download cell for reference.
> earthkit uses `cdsapi` internally — same credentials, same datasets, cleaner interface.


In [ ]:
import earthkit.data as ekd
import earthkit.plots as ekp
import cdsapi  # used internally by earthkit; kept for reference
from pathlib import Path
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

from _utils import get_data_dir
DATA_DIR = get_data_dir()
print('Data cache:', DATA_DIR)
print('earthkit-data:', ekd.__version__)


## 2) Download ERA5: 2m temperature over Africa

We download one month of daily-mean 2m temperature for sub-Saharan Africa.  
This is the kind of data used for **heatwave early warning** climatologies.

Two access patterns shown:
- **`cdsapi` directly**: the traditional approach
- **`earthkit.data` wrapper**: consistent with other notebooks in this series


In [ ]:
# Download ERA5 via earthkit (primary approach — consistent with A01/A02)
era5_path = DATA_DIR / 'era5_africa_2t_2024_jan.grib'

if not era5_path.exists():
    print('Downloading ERA5... (first run, ~1-2 min)')
    ds_era5 = ekd.from_source(
        'cds',
        'reanalysis-era5-single-levels',
        {
            'product_type': 'reanalysis',
            'variable': ['2m_temperature'],
            'year': '2024',
            'month': '01',
            'day': [f'{d:02d}' for d in range(1, 32)],
            'time': ['00:00', '06:00', '12:00', '18:00'],
            'area': [40, -20, -40, 55],  # Africa
            'format': 'grib',
        }
    )
    ds_era5.save(str(era5_path))
    print(f'Saved: {era5_path.name} ({era5_path.stat().st_size/1e6:.1f} MB)')
else:
    print(f'Cached: {era5_path.name} ({era5_path.stat().st_size/1e6:.1f} MB)')

# ── Alternative: native cdsapi (same result, more explicit) ──────────────────
# import cdsapi
# c = cdsapi.Client()
# c.retrieve('reanalysis-era5-single-levels',
#     {'product_type': 'reanalysis', 'variable': '2m_temperature',
#      'year': '2024', 'month': '01', 'day': '15', 'time': '12:00',
#      'area': [40, -20, -40, 55], 'format': 'grib'},
#     str(era5_path))
# cdsapi is the native CDS API client — earthkit uses it internally.
# Use cdsapi directly if you need more control over requests or want to avoid earthkit.


## 3) Open and inspect with earthkit.data


In [ ]:
ds = ekd.from_source('file', str(era5_path))
print(ds.ls())


## 4) Plot: 2m temperature, Africa


In [ ]:
# Select a single 2t field and plot with earthkit.plots
t2m_fields = ds.sel(shortName='2t')
t2m_first = t2m_fields[0]

m = ekp.Map()
m.plot(t2m_first)
m.title(f"ERA5 2m Temperature: Africa (Jan 2024)")


## 5) ERA5 vs real-time Open Data: when to use which

| | ERA5 | ECMWF Open Data |
|---|---|---|
| Time coverage | 1940–present | Last ~4 days |
| Latency | ~5 days lag | ~1–6 hours |
| Grid | 0.25° | 0.25° (9km later 2026) |
| Credentials | CDS free account | None |
| Use case | Climatology, model training | Operational forecasting |

For **IbF model training**: ERA5 provides the historical climate baseline.  
For **operational triggers**: use ECMWF Open Data or SOFF forecasts.


## Take-home messages

- ERA5 is free via CDS: `cdsapi` or `earthkit.data` wrapper, same data
- The earthkit API is consistent: `ekd.from_source('cds', ...)` reads CDS data just like `ekd.from_source('file', ...)`
- Africa bounding box: `area=[40, -20, -40, 55]` (N, W, S, E)
- Next: **B02** for seasonal forecasts, **B03** for CAMS dust
